# Notebook 15: Deep Learning Foundations
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Implement a perceptron from scratch and understand its limits
- Explain and visualise common activation functions
- Trace a forward pass through a multi-layer network
- Implement backpropagation manually on a tiny network
- Know where deep learning excels over classical ML

## 1. The Perceptron

Artificial neurons are inspired by biological ones. A **perceptron**:
1. Receives inputs $x_1, \ldots, x_n$ with weights $w_i$
2. Computes weighted sum + bias: $z = \sum w_i x_i + b$
3. Applies a step function: $\hat{y} = 1$ if $z \geq 0$ else $0$

The **perceptron learning rule**:
$$w_i \leftarrow w_i + \eta (y - \hat{y}) x_i$$
If the prediction is correct, weights don't change. If wrong, they update to push $z$ in the right direction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

class Perceptron:
    def __init__(self, lr=0.1, n_epochs=50):
        self.lr = lr; self.n_epochs = n_epochs

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.errors = []
        for _ in range(self.n_epochs):
            errs = 0
            for xi, yi in zip(X, y):
                pred = self._predict(xi)
                update = self.lr * (yi - pred)
                self.w += update * xi
                self.b += update
                errs += int(update != 0)
            self.errors.append(errs)

    def _predict(self, x): return 1 if np.dot(self.w, x) + self.b >= 0 else 0
    def predict(self, X): return np.array([self._predict(x) for x in X])

# AND gate
X_and = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_and = np.array([0, 0, 0, 1])

p = Perceptron(lr=0.1, n_epochs=20)
p.fit(X_and, y_and)
print('AND gate predictions:')
for xi, yi in zip(X_and, y_and):
    pred = p._predict(xi)
    print(f'  {xi.astype(int)} → pred={pred}, true={yi} {"✓" if pred==yi else "✗"}')

fig, ax = plt.subplots(figsize=(7,3))
ax.plot(p.errors, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Misclassifications')
ax.set_title('Perceptron Learning Curve (AND gate)')
plt.tight_layout(); plt.show()

## 2. The XOR Problem — Why We Need Multiple Layers

A single perceptron can only learn **linearly separable** functions. XOR is the classic counter-example.

Minsky & Papert (1969) proved this limitation, leading to the first AI Winter. The solution: **Multi-Layer Perceptrons (MLP)** trained with backpropagation.

In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

p_xor = Perceptron(lr=0.1, n_epochs=100)
p_xor.fit(X_xor, y_xor)
print('XOR — perceptron still fails:')
for xi, yi, pred in zip(X_xor, y_xor, p_xor.predict(X_xor)):
    print(f'  {xi.astype(int)} → pred={pred}, true={yi} {"✓" if pred==yi else "✗"}')
print(f'Final errors: {p_xor.errors[-1]} (should be > 0)')

## 3. Activation Functions

Activation functions introduce **non-linearity**. Without them, stacking layers collapses to a single linear transformation.

| Function | Formula | Range | Common use |
|----------|---------|-------|------------|
| **Sigmoid** | $1/(1+e^{-z})$ | (0,1) | Binary output |
| **Tanh** | $(e^z-e^{-z})/(e^z+e^{-z})$ | (-1,1) | Older hidden layers |
| **ReLU** | $\max(0,z)$ | $[0,\infty)$ | Default hidden layer |
| **Leaky ReLU** | $\max(0.01z, z)$ | $\mathbb{R}$ | Fixes dying ReLU |
| **Softmax** | $e^{z_k}/\sum e^{z_j}$ | $(0,1)$ | Multiclass output |

In [ ]:
z = np.linspace(-5, 5, 300)
activations = {
    'Sigmoid'    : 1/(1+np.exp(-z)),
    'Tanh'       : np.tanh(z),
    'ReLU'       : np.maximum(0, z),
    'Leaky ReLU' : np.where(z>=0, z, 0.01*z),
    'Swish'      : z/(1+np.exp(-z)),
}
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, (name, fz) in zip(axes, activations.items()):
    ax.plot(z, fz, linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name); ax.set_xlabel('z')
plt.suptitle('Activation Functions')
plt.tight_layout(); plt.show()

## 4. Multi-Layer Perceptron — Forward Pass

Stack layers: Input → Hidden(s) → Output

$$\mathbf{a}^{(1)} = \text{ReLU}(W^{(1)}\mathbf{x} + \mathbf{b}^{(1)})$$
$$\mathbf{a}^{(2)} = \text{ReLU}(W^{(2)}\mathbf{a}^{(1)} + \mathbf{b}^{(2)})$$
$$\hat{y} = \sigma(W^{(3)}\mathbf{a}^{(2)} + b^{(3)})$$

In [ ]:
def relu(z): return np.maximum(0, z)
def softmax(z): e=np.exp(z-z.max()); return e/e.sum()

np.random.seed(1)
W1 = np.random.randn(4,3)*0.5; b1 = np.zeros(4)
W2 = np.random.randn(4,4)*0.5; b2 = np.zeros(4)
W3 = np.random.randn(2,4)*0.5; b3 = np.zeros(2)

def forward(x):
    a1 = relu(W1 @ x + b1)
    a2 = relu(W2 @ a1 + b2)
    out = softmax(W3 @ a2 + b3)
    return out, a1, a2

xin = np.array([1.0, -0.5, 2.0])
out, a1, a2 = forward(xin)
print('Input        :', xin)
print('Hidden 1 act :', a1.round(3))
print('Hidden 2 act :', a2.round(3))
print('Output probs :', out.round(3))
print('Predicted    :', np.argmax(out))

## 5. Backpropagation — Learning Weights

**Backpropagation** applies the **chain rule** to propagate gradients from output back to each weight:

$$\frac{\partial L}{\partial W^{(1)}} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial a^{(2)}} \cdot \frac{\partial a^{(2)}}{\partial a^{(1)}} \cdot \frac{\partial a^{(1)}}{\partial W^{(1)}}$$

Modern frameworks (PyTorch, TensorFlow) compute this automatically via **autograd**.

In [ ]:
# Manual backprop on XOR — 2-in, 2-hidden, 1-out MLP
def sigmoid(z): return 1/(1+np.exp(-z))
def sigmoid_d(a): return a*(1-a)
def bce(y, yh): return -np.mean(y*np.log(yh+1e-8)+(1-y)*np.log(1-yh+1e-8))

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)

np.random.seed(1)
W1 = np.random.randn(2,2)*0.5; b1 = np.zeros((1,2))
W2 = np.random.randn(2,1)*0.5; b2 = np.zeros((1,1))
lr = 0.5; hist = []

for epoch in range(5000):
    a1 = sigmoid(X @ W1 + b1)
    a2 = sigmoid(a1 @ W2 + b2)
    d2 = (a2-y)*sigmoid_d(a2)
    W2 -= lr * a1.T @ d2;  b2 -= lr * d2.mean(0)
    d1 = (d2 @ W2.T)*sigmoid_d(a1)
    W1 -= lr * X.T @ d1;   b1 -= lr * d1.mean(0)
    if epoch % 100 == 0: hist.append(bce(y, a2))

print('XOR after training:')
for xi, yi, pi in zip(X, y, a2):
    print(f'  {xi.astype(int)} → p={pi[0]:.3f}, pred={int(pi[0]>0.5)}, true={int(yi[0])}')

fig, ax = plt.subplots(figsize=(8,3))
ax.plot(hist)
ax.set_xlabel('100-epoch intervals'); ax.set_ylabel('Loss')
ax.set_title('MLP Solving XOR via Backpropagation')
plt.tight_layout(); plt.show()

## Exercises

1. Modify the Perceptron to learn the OR gate. How many epochs does it need?
2. Add a momentum term to the backprop network. Does it converge faster?
3. Plot the decision boundary of the trained XOR network by evaluating it on a 2D grid.
4. Prove algebraically that stacking two linear layers (no activation) is equivalent to a single linear layer.

## Reflection Questions
- What is the "dying ReLU" problem, and how does Leaky ReLU address it?
- Backpropagation requires differentiable activations. How do we handle ReLU, which is non-differentiable at z=0?
- Deep learning models have millions of parameters. Why don't they always overfit?

---
**Next →** `16_pytorch_basics.ipynb`